# SEC MAIS — Teste e Inspeção Bronze: Pé-de-Meia

Este notebook inspeciona os dados brutos extraídos do programa **Pé-de-Meia** armazenados em `armazenamento/bronze/pe_de_meia/`.

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# Localiza o diretório do Pé-de-Meia na camada bronze
def obter_caminho_pe_de_meia() -> Path:
    candidatos = [
        Path.cwd() / "sec-mais" / "armazenamento" / "bronze" / "pe_de_meia",
        Path.cwd().parent / "sec-mais" / "armazenamento" / "bronze" / "pe_de_meia",
        Path.cwd() / "armazenamento" / "bronze" / "pe_de_meia",
        Path.cwd().parent / "armazenamento" / "bronze" / "pe_de_meia",
    ]
    for c in candidatos:
        if c.exists():
            return c.resolve()
    return (Path.cwd() / "sec-mais" / "armazenamento" / "bronze" / "pe_de_meia").resolve()

caminho_pdm = obter_caminho_pe_de_meia()
print(f"📁 Caminho Pé-de-Meia: {caminho_pdm}")
arquivos = list(caminho_pdm.glob("**/*.parquet"))
print(f"📄 Arquivos Parquet encontrados: {len(arquivos)}")
for arq in arquivos:
    print(f"   - {arq.name} ({round(arq.stat().st_size / 1024, 2)} KB)")

📁 Caminho Pé-de-Meia: C:\Users\leonardo.fernandez\Desktop\FERNANDEZ\SEC MAIS\sec-mais\armazenamento\bronze\pe_de_meia
📄 Arquivos Parquet encontrados: 1
   - pe_de_meia_20261111_235421.parquet (3.58 KB)


## 1. Carregamento dos Dados via Pandas
Carregamos o arquivo mais recente ou todos os arquivos particionados.

In [2]:
if arquivos:
    # Ordena pelo mais recente
    arquivo_mais_recente = sorted(arquivos, key=lambda f: f.stat().st_mtime, reverse=True)[0]
    print(f"Lendo arquivo: {arquivo_mais_recente.name}")
    
    df = pd.read_parquet(arquivo_mais_recente)
    display(df.head(10))
else:
    print("⚠️ Nenhum arquivo parquet encontrado. Execute a extração primeiro:\npython -m extracao.programas.pe_de_meia")
    df = pd.DataFrame()

Lendo arquivo: pe_de_meia_20261111_235421.parquet


,cd_beneficiario,nm_beneficiario,dt_referencia,_data_extracao,_programa
0,1,BENEFICIARIO EXEMPLO,2026-11-11,2026-11-11 23:54:21.845772,pe_de_meia


## 2. Estrutura, Tipagem e Verificação de Nulos

In [3]:
if not df.empty:
    print(f"Total de Linhas: {len(df)}")
    print(f"Total de Colunas: {len(df.columns)}\n")
    
    info_colunas = pd.DataFrame({
        "Tipo": df.dtypes,
        "Valores Não-Nulos": df.notnull().sum(),
        "Nulos": df.isnull().sum(),
        "% Nulos": (df.isnull().sum() / len(df) * 100).round(2),
        "Valores Únicos": df.nunique()
    })
    display(info_colunas)
else:
    print("Sem dados para analisar.")

Total de Linhas: 1
Total de Colunas: 5



,Tipo,Valores Não-Nulos,Nulos,% Nulos,Valores Únicos
cd_beneficiario,int64,1,0,0.0,1
nm_beneficiario,str,1,0,0.0,1
dt_referencia,object,1,0,0.0,1
_data_extracao,datetime64[us],1,0,0.0,1
_programa,str,1,0,0.0,1


## 3. Validação de Metadados Técnicos (`_data_extracao`, `_programa`)

In [4]:
if not df.empty:
    cols_tecnicas = [c for c in df.columns if c.startswith("_")]
    print(f"Colunas técnicas de controle: {cols_tecnicas}")
    if cols_tecnicas:
        display(df[cols_tecnicas].drop_duplicates())
else:
    print("Sem dados para analisar.")

Colunas técnicas de controle: ['_data_extracao', '_programa']


,_data_extracao,_programa
0,2026-11-11 23:54:21.845772,pe_de_meia


## 4. Consulta SQL Rápida com DuckDB (Opcional)

In [5]:
try:
    import duckdb
    if arquivos:
        # Lê diretamente o arquivo parquet sem carregar no pandas antes
        caminho_sql = str(arquivo_mais_recente).replace("\\", "/")
        resultado = duckdb.query(f"SELECT * FROM read_parquet('{caminho_sql}') LIMIT 5").df()
        print("✅ Leitura via DuckDB concluída com sucesso:")
        display(resultado)
except ImportError:
    print("DuckDB não está instalado neste ambiente virtual. (Opcional)")

✅ Leitura via DuckDB concluída com sucesso:


,cd_beneficiario,nm_beneficiario,dt_referencia,_data_extracao,_programa,ano,dia,mes
0,1,BENEFICIARIO EXEMPLO,2026-11-11,2026-11-11 23:54:21.845772,pe_de_meia,2026,11,11
